# WAV Marker Processing Script: Top-Level Description

This script processes breathing guide audio files by encoding inhale and exhale markers directly into WAV files. Here's what it does, from top to bottom:

## Configuration Layer
- Sets a repeat count for how many times to append the second audio segment
- Defines a simple binary classification system: odd IDs for inhale events, even IDs for exhale events

## Data Acquisition Layer
1. **Reads source files**:
   - Loads two WAV audio files (intro and repetition segment)
   - Reads two CSV files containing timestamp information
   - Parses the CSV files to identify breathing phases at specific timestamps

## Marker Encoding Layer
1. **Embeds binary markers in WAV files**:
   - Creates two new WAV files with embedded markers
   - Uses odd numbers (1,3,5...) to mark inhale events
   - Uses even numbers (2,4,6...) to mark exhale events
   - Maps each timestamp to an exact sample position in the audio

## Audio Manipulation Layer
1. **Processes audio segments**:
   - Concatenates the first audio segment with multiple copies of the second segment
   - Maintains proper timing and alignment throughout concatenation
   - Creates a single, longer WAV file with all audio content

## Marker Management Layer
1. **Handles markers during concatenation**:
   - Adjusts timestamps from repeated segments based on their new position
   - Maintains the correct order and timing of all markers
   - Ensures all markers from all segments are preserved

## Extraction & Verification Layer
1. **Reads markers from the concatenated file**:
   - Extracts all markers directly from the WAV file's metadata
   - Identifies each marker's type (inhale/exhale) based on odd/even ID
   - Calculates precise timestamp information for each marker

## Output Generation Layer
1. **Creates the final output**:
   - Produces a concatenated WAV file with all markers embedded
   - Generates a CSV file with all marker information
   - Maintains the same column format as the original CSVs

The script is designed to be robust, handling any number of markers and repetitions while preserving the exact timing and classification of each breathing event throughout the entire process.

In [10]:
import wave
import numpy as np
import pandas as pd
import os
import struct
from pydub import AudioSegment
import io

# =============================================
# TOP-LEVEL CONFIGURATION
# =============================================
# How many times to repeat the second audio segment
REPEAT_COUNT = 5  # Change this to control repetition
# =============================================

# Marker type constants - using odd/even for unlimited markers
MARKER_TYPE_INHALATION = 1  # Will use odd IDs: 1, 3, 5, 7, ...
MARKER_TYPE_EXHALATION = 2  # Will use even IDs: 2, 4, 6, 8, ...

def read_markers_from_csv(csv_file):
    """Read marker timestamps and breathing phase info from a CSV file"""
    df = pd.read_csv(csv_file)
    print(f"CSV columns: {df.columns.tolist()}")
    
    markers = []
    
    # Check for breathphase column and expected format
    if 'breathphase' in df.columns:
        for _, row in df.iterrows():
            # Determine marker type based on breathphase
            marker_type = None
            if 'inhale' in str(row['breathphase']).lower():
                marker_type = MARKER_TYPE_INHALATION
            elif 'exhale' in str(row['breathphase']).lower():
                marker_type = MARKER_TYPE_EXHALATION
            else:
                print(f"Warning: Unknown breathing phase: {row['breathphase']}")
                continue  # Skip this marker if we can't classify it
            
            # Get required timestamp information
            marker = {
                'timestamp': row['milliseconds'] / 1000.0,  # Convert to seconds for WAV cue points
                'breathphase': row['breathphase'],
                'marker_type': marker_type
            }
            
            # Copy all CSV columns for later reference
            for col in df.columns:
                if col not in marker:
                    marker[col] = row[col]
            
            # Print a few examples of the original retentionslot_timestamp format
            if len(markers) < 3:
                print(f"Original retentionslot_timestamp format: {row['retentionslot_timestamp']}")
                    
            markers.append(marker)
    else:
        print(f"Error: CSV file {csv_file} does not contain a 'breathphase' column")
        return [], df.columns.tolist()
    
    # Sort markers by timestamp
    markers.sort(key=lambda x: x['timestamp'])
    return markers, df.columns.tolist()

def add_markers_to_wav(wav_file, markers, output_file):
    """Add simple inhale/exhale markers directly to WAV file using odd/even IDs"""
    # Read audio file
    audio = AudioSegment.from_wav(wav_file)
    sample_rate = audio.frame_rate
    
    # Create cue points list for the WAV file - ONLY encoding the breathphase
    cue_points = []
    
    # Write debug info
    print(f"Adding markers to {output_file}:")
    print(f"  Sample rate: {sample_rate} Hz")
    
    # Current ID counters for each type
    inhale_id = 1  # Starting with 1 (odd)
    exhale_id = 2  # Starting with 2 (even)
    
    for i, marker in enumerate(markers):
        # Convert timestamp to sample position
        sample_pos = int(marker['timestamp'] * sample_rate)
        
        # Get marker type (inhale or exhale) and assign ID
        marker_type = marker.get('marker_type')
        if marker_type == MARKER_TYPE_INHALATION:
            cue_id = inhale_id
            inhale_id += 2  # Next odd number
        else:  # MARKER_TYPE_EXHALATION
            cue_id = exhale_id
            exhale_id += 2  # Next even number
        
        # Debug output
        if i < 5 or i > len(markers) - 5 or i % 100 == 0:  # Show first/last 5 markers and every 100th
            print(f"  Marker {i+1}: ID={cue_id}, Position={sample_pos}, Type={'Inhale' if marker_type == MARKER_TYPE_INHALATION else 'Exhale'}")
        
        # Add to cue points - just the binary classification info
        cue_points.append((cue_id, sample_pos, 0, 0, 0, 0, 0, 0))
    
    # Save to WAV with embedded cue points
    _export_wav_with_markers(audio, output_file, cue_points)
    
    print(f"Created {output_file} with {len(markers)} embedded markers")
    return output_file

def _export_wav_with_markers(audio, output_file, cue_points):
    """Export a WAV file with embedded cue points"""
    # First export the audio to a temporary file
    temp_file = io.BytesIO()
    audio.export(temp_file, format="wav")
    temp_file.seek(0)
    
    # Read the WAV data
    with wave.open(temp_file, 'rb') as wf:
        params = wf.getparams()
        frames = wf.readframes(wf.getnframes())
    
    # Write the WAV file with cue points
    with wave.open(output_file, 'wb') as wf:
        wf.setparams(params)
        wf.writeframes(frames)
    
    # Add cue chunk if we have cue points
    if cue_points:
        try:
            # We need to manually add the cue chunk
            with open(output_file, 'rb+') as f:
                # Go to the end of the file
                f.seek(0, 2)
                file_size = f.tell()
                
                # Create cue chunk
                cue_data = struct.pack('<L', len(cue_points))  # Number of cue points
                
                for cue_id, position, chunk_start, block_start, sample_offset, _, _, _ in cue_points:
                    # Pack the cue point data
                    cue_data += struct.pack('<L', cue_id)  # cue ID
                    cue_data += struct.pack('<L', position)  # play position
                    cue_data += b'data'  # data chunk ID
                    cue_data += struct.pack('<L', 0)  # chunk start
                    cue_data += struct.pack('<L', 0)  # block start
                    cue_data += struct.pack('<L', 0)  # sample offset
                
                # Write cue chunk header
                f.write(b'cue ')
                f.write(struct.pack('<L', len(cue_data)))  # chunk size
                f.write(cue_data)  # chunk data
                
                # Update RIFF chunk size
                end_pos = f.tell()
                f.seek(4)
                f.write(struct.pack('<L', end_pos - 8))  # RIFF size = file size - 8
                
                print(f"Added cue chunk with {len(cue_points)} markers, total file size: {end_pos} bytes")
        except Exception as e:
            print(f"Error adding cue chunk: {e}")
            import traceback
            traceback.print_exc()
    
    # Verify the file has the expected markers
    try:
        with open(output_file, 'rb') as f:
            # Check for cue chunk
            f.seek(12)  # Skip RIFF header and file size
            
            while True:
                chunk_pos = f.tell()
                chunk_id = f.read(4)
                
                if not chunk_id or len(chunk_id) < 4:
                    break
                
                chunk_size = int.from_bytes(f.read(4), byteorder='little')
                
                if chunk_id == b'cue ':
                    num_cues = int.from_bytes(f.read(4), byteorder='little')
                    print(f"Verified: WAV file has {num_cues} cue points")
                    break
                
                # Skip to next chunk
                f.seek(chunk_pos + 8 + chunk_size)
    except Exception as e:
        print(f"Error verifying cue chunk: {e}")
    
    return output_file

def concatenate_wav_files(wav1, wav2, output_file):
    """Concatenate two WAV files into a new one"""
    # Read the audio files with pydub
    audio1 = AudioSegment.from_wav(wav1)
    audio2 = AudioSegment.from_wav(wav2)
    
    # Get durations in seconds
    duration1 = len(audio1) / 1000.0  # pydub uses milliseconds
    
    # Concatenate the audio
    concatenated = audio1 + audio2
    
    # Export the concatenated audio (without markers for now)
    concatenated.export(output_file, format="wav")
    
    print(f"Created concatenated file: {output_file}")
    return output_file, duration1, audio1.frame_rate

def extract_markers_from_wav(wav_file):
    """Extract marker information directly from a WAV file's cue points using odd/even ID detection"""
    # Read the audio file to get the sample rate
    audio = AudioSegment.from_wav(wav_file)
    sample_rate = audio.frame_rate
    
    markers = []
    
    try:
        with open(wav_file, 'rb') as f:
            # Check RIFF header
            if f.read(4) != b'RIFF':
                raise ValueError("Not a valid RIFF file")
            
            # Read file size (and skip)
            f.read(4)
            
            # Check WAVE format
            if f.read(4) != b'WAVE':
                raise ValueError("Not a valid WAVE file")
            
            # Find all chunks, including potentially multiple cue chunks
            while True:
                chunk_pos = f.tell()
                chunk_header = f.read(4)
                
                if not chunk_header or len(chunk_header) < 4:
                    break  # End of file
                
                chunk_size = int.from_bytes(f.read(4), byteorder='little')
                print(f"Found chunk: {chunk_header.decode('ascii', errors='replace')} size: {chunk_size}")
                
                if chunk_header == b'cue ':
                    # Found cue chunk
                    num_cue_points = int.from_bytes(f.read(4), byteorder='little')
                    print(f"Found {num_cue_points} cue points in WAV file")
                    
                    # Read all cue points
                    for i in range(num_cue_points):
                        cue_chunk_start = f.tell()
                        
                        # Read cue point
                        cue_id_bytes = f.read(4)
                        cue_id = int.from_bytes(cue_id_bytes, byteorder='little')
                        
                        position_bytes = f.read(4)
                        position = int.from_bytes(position_bytes, byteorder='little')
                        
                        data_chunk_id = f.read(4)  # Should be 'data'
                        chunk_start = int.from_bytes(f.read(4), byteorder='little')
                        block_start = int.from_bytes(f.read(4), byteorder='little')
                        sample_offset = int.from_bytes(f.read(4), byteorder='little')
                        
                        if i < 5 or i > num_cue_points - 5 or i % 100 == 0:
                            print(f"  Cue point {i+1}: ID={cue_id}, Position={position}, DataChunk={data_chunk_id}")
                        
                        # Determine marker type from cue ID parity (odd/even)
                        marker_type = None
                        breathphase = None
                        
                        if cue_id % 2 == 1:  # Odd ID = Inhale
                            marker_type = MARKER_TYPE_INHALATION
                            breathphase = "inhale"
                        elif cue_id % 2 == 0:  # Even ID = Exhale
                            marker_type = MARKER_TYPE_EXHALATION
                            breathphase = "exhale"
                        else:
                            # This condition can't actually happen (every number is either odd or even),
                            # but kept for consistency with previous logic
                            print(f"  Skipping cue ID {cue_id} - unexpected value")
                            continue
                        
                        # Calculate timestamp
                        timestamp_seconds = position / sample_rate
                        milliseconds = int(timestamp_seconds * 1000)
                        
                        # Format retentionslot_timestamp in mm.ss.ms format
                        minutes = int(timestamp_seconds // 60)
                        seconds = int(timestamp_seconds % 60)
                        ms = int((timestamp_seconds % 1) * 1000)
                        retentionslot_timestamp = f"{minutes:02d}.{seconds:02d}.{ms:03d}"
                        
                        # Create marker entry with timestamps in required formats
                        marker = {
                            'retentionslot_timestamp': retentionslot_timestamp,
                            'milliseconds': milliseconds,
                            'sample_index': position,
                            'description': "",  # Empty as requested
                            'breathphase': breathphase,
                            'marker_type': marker_type
                        }
                        
                        markers.append(marker)
                    
                    # Continue searching for more cue chunks (don't break)
                    f.seek(chunk_pos + 8 + chunk_size)
                else:
                    # Skip this chunk
                    f.seek(chunk_size, 1)  # 1 means relative to current position
        
    except Exception as e:
        print(f"Error reading WAV file: {e}")
        import traceback
        traceback.print_exc()
    
    # Sort markers by timestamp
    markers.sort(key=lambda x: x['milliseconds'])
    
    if not markers:
        print("No markers found in WAV file")
    else:
        print(f"Extracted {len(markers)} markers from WAV file")
        print("Sample markers:")
        for i in range(min(5, len(markers))):
            print(f"  {i+1}: time={markers[i]['retentionslot_timestamp']}, phase={markers[i]['breathphase']}")
        
        # Print some middle and end markers if there are many
        if len(markers) > 20:
            middle_idx = len(markers) // 2
            print("  ...")
            print(f"  {middle_idx}: time={markers[middle_idx-1]['retentionslot_timestamp']}, phase={markers[middle_idx-1]['breathphase']}")
            print("  ...")
            for i in range(max(0, len(markers)-5), len(markers)):
                print(f"  {i+1}: time={markers[i]['retentionslot_timestamp']}, phase={markers[i]['breathphase']}")
    
    return markers

def adjust_markers_for_concatenated_file(markers1, markers2, duration1, sample_rate):
    """
    Adjust markers from the second file by adding the duration of the first file
    This accounts for the concatenation
    """
    # Adjust timestamps from the second file
    adjusted_markers2 = []
    for marker in markers2:
        adjusted_marker = marker.copy()
        
        # Calculate new timestamp in seconds
        adjusted_timestamp = marker['timestamp'] + duration1
        
        # Update all timestamp fields
        adjusted_marker['timestamp'] = adjusted_timestamp
        adjusted_marker['milliseconds'] = int(adjusted_timestamp * 1000)
        adjusted_marker['sample_index'] = int(adjusted_timestamp * sample_rate)
        
        adjusted_markers2.append(adjusted_marker)
    
    # Combine all markers and sort by timestamp
    all_markers = markers1 + adjusted_markers2
    all_markers.sort(key=lambda x: x['timestamp'])
    return all_markers

def export_markers_to_csv(markers, output_csv, column_format):
    """Export markers to a CSV file in the specified format"""
    # Create a DataFrame with the specified columns
    df_data = []
    
    for marker in markers:
        row = {}
        for col in column_format:
            row[col] = marker.get(col, "")
        df_data.append(row)
    
    # Create DataFrame
    df = pd.DataFrame(df_data, columns=column_format)
    
    # Save to CSV
    df.to_csv(output_csv, index=False)
    print(f"Exported {len(markers)} markers to {output_csv}")
    return output_csv

def main():
    # File paths from user input
    wav_file1 = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_intro.wav"
    wav_file2 = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_repetitionsegment.wav"
    csv_file1 = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_timestamps.csv"
    csv_file2 = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_repetition_timestamps.csv"
    
    # Output file paths with repeat count in the name
    output_dir = os.path.dirname(wav_file1)
    wav_with_markers1 = os.path.join(output_dir, "BoxBreathing_intro_with_markers.wav")
    wav_with_markers2 = os.path.join(output_dir, "BoxBreathing_repetitionsegment_with_markers.wav")
    
    # Add repeat count to output filenames if more than 1 repeat
    repeat_suffix = f"_x{REPEAT_COUNT}" if REPEAT_COUNT > 1 else ""
    concatenated_wav = os.path.join(output_dir, f"BoxBreathing_complete{repeat_suffix}.wav")
    output_csv = os.path.join(output_dir, f"BoxBreathing_complete{repeat_suffix}_markers.csv")
    
    print(f"Configuration: Repeating second segment {REPEAT_COUNT} times")
    
    # STEP 1: Read markers from CSV files
    print(f"Reading markers from {csv_file1}...")
    markers1, columns1 = read_markers_from_csv(csv_file1)
    print(f"Found {len(markers1)} markers in first CSV")
    
    print(f"Reading markers from {csv_file2}...")
    markers2, columns2 = read_markers_from_csv(csv_file2)
    print(f"Found {len(markers2)} markers in second CSV")
    
    # Get the CSV column format
    csv_columns = columns1 if 'breathphase' in columns1 else columns2
    print(f"Using CSV format: {csv_columns}")
    
    # STEP 2: Create new WAV files with embedded markers
    print(f"Creating new WAV file with markers: {wav_with_markers1}")
    add_markers_to_wav(wav_file1, markers1, wav_with_markers1)
    
    print(f"Creating new WAV file with markers: {wav_with_markers2}")
    add_markers_to_wav(wav_file2, markers2, wav_with_markers2)
    
    # STEP 3: Concatenate the WAV files with repetition
    print(f"Concatenating WAV files to: {concatenated_wav} (with {REPEAT_COUNT} repeats)")
    
    # Read the first audio segment
    audio1 = AudioSegment.from_wav(wav_with_markers1)
    audio2 = AudioSegment.from_wav(wav_with_markers2)
    
    # Get duration of first segment
    duration1 = len(audio1) / 1000.0  # pydub uses milliseconds
    duration2 = len(audio2) / 1000.0
    
    sample_rate = audio1.frame_rate
    
    # Start with the first segment
    concatenated = audio1
    
    # Add repeated instances of the second segment
    for i in range(REPEAT_COUNT):
        print(f"  Adding repeat {i+1} of {REPEAT_COUNT}...")
        concatenated += audio2
    
    # Export the concatenated audio
    concatenated.export(concatenated_wav, format="wav")
    print(f"Created concatenated file: {concatenated_wav}")
    
    # STEP 4: Combine and adjust all markers
    print(f"Adjusting markers for concatenated file with {REPEAT_COUNT} repeats...")
    
    # Start with markers from the first file
    all_markers = markers1.copy()
    
    # Add markers from each repeat of the second file
    current_offset = duration1
    for i in range(REPEAT_COUNT):
        # Adjust markers from this repeat
        for marker in markers2:
            adjusted_marker = marker.copy()
            
            # Calculate new timestamp in seconds
            adjusted_timestamp = marker['timestamp'] + current_offset
            
            # Update all timestamp fields
            adjusted_marker['timestamp'] = adjusted_timestamp
            adjusted_marker['milliseconds'] = int(adjusted_timestamp * 1000)
            adjusted_marker['sample_index'] = int(adjusted_timestamp * sample_rate)
            
            all_markers.append(adjusted_marker)
        
        # Update the offset for the next repeat
        current_offset += duration2
    
    # Sort all markers by timestamp
    all_markers.sort(key=lambda x: x['timestamp'])
    
    print(f"Total markers after repetition: {len(all_markers)}")
    
    # STEP 5: Add all markers to the concatenated file
    print(f"Adding all markers to concatenated file...")
    add_markers_to_wav(concatenated_wav, all_markers, concatenated_wav)
    
    # STEP 6: Extract markers directly from the concatenated WAV file
    print(f"Extracting markers from concatenated WAV file...")
    extracted_markers = extract_markers_from_wav(concatenated_wav)
    
    # STEP 7: Export the extracted markers to a CSV in the original format
    print(f"Exporting extracted markers to CSV: {output_csv}")
    export_markers_to_csv(extracted_markers, output_csv, csv_columns)
    
    # Print summary
    print("\nProcess complete!")
    print(f"Created WAV files with embedded markers:")
    print(f"  - {wav_with_markers1}")
    print(f"  - {wav_with_markers2}")
    print(f"Concatenated file with all markers ({REPEAT_COUNT} repeats):")
    print(f"  - {concatenated_wav}")
    print(f"Extracted markers to CSV:")
    print(f"  - {output_csv}")
    
    # Calculate expected duration
    total_duration = duration1 + (duration2 * REPEAT_COUNT)
    minutes = int(total_duration // 60)
    seconds = int(total_duration % 60)
    print(f"Total audio duration: {minutes}m {seconds}s ({total_duration:.2f} seconds)")
    
    # Print marker summary
    inhalation_count = sum(1 for m in extracted_markers if m.get('marker_type') == MARKER_TYPE_INHALATION)
    exhalation_count = sum(1 for m in extracted_markers if m.get('marker_type') == MARKER_TYPE_EXHALATION)
    
    print("\nExtracted breathing phase marker summary:")
    print(f"  Inhale markers: {inhalation_count}")
    print(f"  Exhale markers: {exhalation_count}")
    print(f"  Total markers: {len(extracted_markers)}")
    print(f"  Expected markers: {len(markers1) + (len(markers2) * REPEAT_COUNT)}")
    
    if len(extracted_markers) != len(markers1) + (len(markers2) * REPEAT_COUNT):
        print("WARNING: Number of extracted markers does not match expected count!")
        print("         This might indicate some markers were not properly embedded or extracted.")

if __name__ == "__main__":
    main()

Configuration: Repeating second segment 5 times
Reading markers from C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_timestamps.csv...
CSV columns: ['retentionslot_timestamp', 'milliseconds', 'sample_index', 'description', 'breathphase']
Original retentionslot_timestamp format: 00:31.9
Original retentionslot_timestamp format: 00:39.9
Original retentionslot_timestamp format: 00:48.0
Found 38 markers in first CSV
Reading markers from C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_repetition_timestamps.csv...
CSV columns: ['retentionslot_timestamp', 'milliseconds', 'sample_index', 'description', 'breathphase']
Original retentionslot_timestamp format: 00:07.0
Original retentionslot_timestamp format: 00:14.9
Original retentionslot_timestamp format: 00:23.0
Found 34 markers in second CSV
Using CSV format: ['retentionslot_timestamp', 'milliseconds', 'sample_index', 'description', 'breathphase']
Creating new WAV f